# Day 2 - Session 3: End-to-End Machine Learning
## Credit Scoring Model untuk Kredit UMKM

**Workshop: Snowflake x Bank**

Pada notebook ini kita akan membangun **Credit Scoring Model** end-to-end:

1. **Part 1:** Data Preparation & EDA
2. **Part 2:** Feature Engineering & Feature Store
3. **Part 3:** Model Training (XGBoost Credit Scoring)
4. **Part 4:** Model Registry & Deployment
5. **Part 5:** Batch Inference & Explainability (SHAP)
6. **Part 6:** ML Observability & Monitoring

**Use Case:** Prediksi risiko kredit nasabah UMKM - apakah kredit akan *DEFAULT* atau *NON-DEFAULT*

> Sangat relevan untuk Bank Mandiri yang fokus pada lending UMKM via Livin' Merchant (cash-flow based lending)

---
## Setup: Koneksi ke Snowflake & Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.functions import col, lit, when, avg, sum as sum_, count, max as max_, min as min_, datediff

import pandas as pd
import numpy as np

# NOTE: If running inside a Snowflake Workspace Notebook, the session is provided automatically
# via get_active_session() and the two lines below are NOT needed.
# Uncomment them only when running locally (VS Code / local Jupyter) with a connections.toml entry.
# connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME", "default")
# session = Session.builder.config("connection_name", connection_name).create()

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE WAREHOUSE BANK_WH").collect()
session.sql("USE DATABASE BANK_DB").collect()
session.sql("USE SCHEMA RAW_DATA").collect()

print(f"Connected: {session.get_current_account()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

---
## Part 1: Data Preparation & Exploratory Data Analysis (EDA)

Kita akan menggunakan data dari Session 1:
- `DIM_NASABAH` - Data demografis nasabah
- `DIM_CABANG` - Data cabang
- `FACT_KREDIT` - Data kredit (mengandung target: `STATUS_KOLEKTIBILITAS`)
- `FACT_TRANSAKSI` - Data transaksi
- `FACT_SIMPANAN` - Data simpanan

In [ ]:
df_nasabah = session.table("BANK_DB.RAW_DATA.DIM_NASABAH")
df_cabang = session.table("BANK_DB.RAW_DATA.DIM_CABANG")
df_kredit = session.table("BANK_DB.RAW_DATA.FACT_KREDIT")
df_transaksi = session.table("BANK_DB.RAW_DATA.FACT_TRANSAKSI")
df_simpanan = session.table("BANK_DB.RAW_DATA.FACT_SIMPANAN")

print("=== Data Overview ===")
print(f"DIM_NASABAH     : {df_nasabah.count():,} rows")
print(f"DIM_CABANG      : {df_cabang.count():,} rows")
print(f"FACT_KREDIT     : {df_kredit.count():,} rows")
print(f"FACT_TRANSAKSI  : {df_transaksi.count():,} rows")
print(f"FACT_SIMPANAN   : {df_simpanan.count():,} rows")

In [ ]:
print("=== Distribusi Status Kolektibilitas (Target Variable) ===")
df_kredit.group_by("STATUS_KOLEKTIBILITAS") \
    .agg(count("*").alias("JUMLAH")) \
    .sort("STATUS_KOLEKTIBILITAS") \
    .show()

print("\n=== Distribusi Jenis Kredit ===")
df_kredit.group_by("JENIS_KREDIT") \
    .agg(count("*").alias("JUMLAH")) \
    .sort(col("JUMLAH").desc()) \
    .show()

print("\n=== Preview FACT_KREDIT ===")
df_kredit.limit(5).show()

### 1.1 Buat Binary Target Variable

Status kolektibilitas 1-5 di-convert ke binary:
- **NON-DEFAULT (0):** Kolektibilitas 1 (Lancar) dan 2 (Dalam Perhatian Khusus)
- **DEFAULT (1):** Kolektibilitas 3 (Kurang Lancar), 4 (Diragukan), 5 (Macet)

> Sesuai ketentuan OJK, kredit bermasalah (NPL) adalah kolektibilitas 3-5

In [ ]:
df_kredit_labeled = df_kredit.with_column(
    "IS_DEFAULT",
    when(col("STATUS_KOLEKTIBILITAS").like("1-%"), lit(0))
    .when(col("STATUS_KOLEKTIBILITAS").like("2-%"), lit(0))
    .otherwise(lit(1))
)

print("=== Distribusi Target (Binary) ===")
df_kredit_labeled.group_by("IS_DEFAULT") \
    .agg(count("*").alias("JUMLAH")) \
    .sort("IS_DEFAULT") \
    .show()

total = df_kredit_labeled.count()
defaults = df_kredit_labeled.filter(col("IS_DEFAULT") == 1).count()
print(f"\nDefault Rate: {defaults/total*100:.1f}%")
print(f"Non-Default: {total-defaults:,} | Default: {defaults:,}")

### 1.2 EDA - Visualisasi

In [ ]:
import matplotlib.pyplot as plt
# %matplotlib inline  # (optional) explicit inline backend for notebooks
# import matplotlib
# matplotlib.use('Agg')   # 'Agg' is non-interactive; plt.show() would not render. Keep disabled in notebook.

pd_kredit = df_kredit_labeled.to_pandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

status_counts = pd_kredit['STATUS_KOLEKTIBILITAS'].value_counts().sort_index()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b', '#8e44ad']
axes[0, 0].bar(range(len(status_counts)), status_counts.values, color=colors[:len(status_counts)])
axes[0, 0].set_xticks(range(len(status_counts)))
axes[0, 0].set_xticklabels([s.split('-')[0] for s in status_counts.index], rotation=0)
axes[0, 0].set_title('Distribusi Kolektibilitas', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Jumlah')

default_counts = pd_kredit['IS_DEFAULT'].value_counts().sort_index()
axes[0, 1].pie(default_counts.values, labels=['Non-Default', 'Default'],
               autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0, 1].set_title('Proporsi Default vs Non-Default', fontsize=12, fontweight='bold')

jenis_default = pd_kredit.groupby('JENIS_KREDIT')['IS_DEFAULT'].mean().sort_values(ascending=False)
axes[1, 0].barh(range(len(jenis_default)), jenis_default.values, color='#3498db')
axes[1, 0].set_yticks(range(len(jenis_default)))
axes[1, 0].set_yticklabels(jenis_default.index, fontsize=9)
axes[1, 0].set_title('Default Rate per Jenis Kredit', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Default Rate')

axes[1, 1].hist(pd_kredit['JUMLAH_PINJAMAN'] / 1e6, bins=30, color='#9b59b6', edgecolor='white')
axes[1, 1].set_title('Distribusi Jumlah Pinjaman', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Jumlah Pinjaman (Juta Rp)')
axes[1, 1].set_ylabel('Frekuensi')

plt.suptitle('EDA - Credit Scoring Bank', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
# plt.savefig('eda_credit_scoring.png', dpi=150, bbox_inches='tight')
plt.show()
# print("Plot saved: eda_credit_scoring.png")

---
## Part 2: Feature Engineering & Feature Store

Kita akan membuat features yang relevan untuk credit scoring perbankan menggunakan **Snowflake Feature Store**.

### Features yang akan dibuat:
| Feature | Deskripsi | Sumber |
|---------|-----------|--------|
| `avg_monthly_trx_amount` | Rata-rata jumlah transaksi bulanan | FACT_TRANSAKSI |
| `total_trx_count` | Total jumlah transaksi | FACT_TRANSAKSI |
| `digital_channel_ratio` | Rasio transaksi via channel digital | FACT_TRANSAKSI |
| `failed_trx_ratio` | Rasio transaksi gagal | FACT_TRANSAKSI |
| `total_simpanan` | Total saldo simpanan | FACT_SIMPANAN |
| `jumlah_produk_simpanan` | Jumlah produk simpanan yang dimiliki | FACT_SIMPANAN |
| `saving_ratio` | Rasio simpanan terhadap pinjaman | FACT_SIMPANAN + FACT_KREDIT |
| `penghasilan_bulanan` | Penghasilan bulanan nasabah | DIM_NASABAH |
| `lama_jadi_nasabah_tahun` | Berapa lama jadi nasabah (tahun) | DIM_NASABAH |
| `credit_utilization_ratio` | Rasio outstanding terhadap plafon | FACT_KREDIT |

### 2.1 Buat Feature Table dari Transaksi

In [ ]:
session.sql("CREATE SCHEMA IF NOT EXISTS BANK_DB.ML_FEATURES COMMENT = 'Schema untuk ML Feature Store'").collect()
session.sql("USE SCHEMA BANK_DB.RAW_DATA").collect()

trx_features = df_transaksi.group_by("NASABAH_ID").agg(
    avg("JUMLAH").alias("AVG_TRX_AMOUNT"),
    count("*").alias("TOTAL_TRX_COUNT"),
    sum_(when(col("CHANNEL").isin("Mobile Banking", "Internet Banking", "QRIS"), lit(1)).otherwise(lit(0))).alias("DIGITAL_TRX_COUNT"),
    sum_(when(col("STATUS") == "Gagal", lit(1)).otherwise(lit(0))).alias("FAILED_TRX_COUNT"),
    sum_("JUMLAH").alias("TOTAL_TRX_VOLUME")
)

trx_features = trx_features.with_column(
    "DIGITAL_CHANNEL_RATIO",
    col("DIGITAL_TRX_COUNT") / col("TOTAL_TRX_COUNT")
).with_column(
    "FAILED_TRX_RATIO",
    col("FAILED_TRX_COUNT") / col("TOTAL_TRX_COUNT")
)

print("=== Transaction Features ===")
trx_features.limit(5).show()

### 2.2 Buat Feature Table dari Simpanan

In [ ]:
simpanan_features = df_simpanan.filter(col("STATUS") == "Aktif").group_by("NASABAH_ID").agg(
    sum_("SALDO").alias("TOTAL_SIMPANAN"),
    count("*").alias("JUMLAH_PRODUK_SIMPANAN"),
    max_("SALDO").alias("MAX_SALDO_SIMPANAN"),
    avg("SUKU_BUNGA_PERSEN").alias("AVG_BUNGA_SIMPANAN")
)

print("=== Simpanan Features ===")
simpanan_features.limit(5).show()

### 2.3 Buat Feature Table dari Nasabah (Demografis)

In [ ]:
nasabah_features = df_nasabah.select(
    col("NASABAH_ID"),
    col("PENGHASILAN_BULANAN"),
    col("SEGMEN_NASABAH"),
    col("PEKERJAAN"),
    col("JENIS_KELAMIN"),
    datediff("year", col("TANGGAL_BUKA_REKENING"), F.current_date()).alias("LAMA_JADI_NASABAH_TAHUN"),
    datediff("year", col("TANGGAL_LAHIR"), F.current_date()).alias("USIA")
)

print("=== Nasabah Features ===")
nasabah_features.limit(5).show()

### 2.4 Gabungkan Semua Features + Register ke Feature Store

In [ ]:
kredit_base = df_kredit_labeled.select(
    col("KREDIT_ID"),
    col("NASABAH_ID"),
    col("JENIS_KREDIT"),
    col("JUMLAH_PINJAMAN"),
    col("OUTSTANDING"),
    col("TENOR_BULAN"),
    col("SUKU_BUNGA_PERSEN"),
    col("SEKTOR_EKONOMI"),
    col("AGUNAN"),
    col("IS_DEFAULT"),
    (col("OUTSTANDING") / col("JUMLAH_PINJAMAN")).alias("CREDIT_UTILIZATION_RATIO")
)

master_features = kredit_base \
    .join(nasabah_features, "NASABAH_ID", "left") \
    .join(trx_features, "NASABAH_ID", "left") \
    .join(simpanan_features, "NASABAH_ID", "left")

master_features = master_features.with_column(
    "SAVING_RATIO",
    when(col("JUMLAH_PINJAMAN") > 0,
         F.coalesce(col("TOTAL_SIMPANAN"), lit(0)) / col("JUMLAH_PINJAMAN"))
    .otherwise(lit(0))
)

master_features = master_features.na.fill({
    "AVG_TRX_AMOUNT": 0,
    "TOTAL_TRX_COUNT": 0,
    "DIGITAL_CHANNEL_RATIO": 0,
    "FAILED_TRX_RATIO": 0,
    "TOTAL_SIMPANAN": 0,
    "JUMLAH_PRODUK_SIMPANAN": 0,
    "SAVING_RATIO": 0,
    "TOTAL_TRX_VOLUME": 0,
    "MAX_SALDO_SIMPANAN": 0,
    "AVG_BUNGA_SIMPANAN": 0,
    "DIGITAL_TRX_COUNT": 0,
    "FAILED_TRX_COUNT": 0
})

print(f"=== Master Feature Table: {master_features.count()} rows ===")
master_features.limit(5).show()

### 2.5 Register ke Snowflake Feature Store

Snowflake Feature Store menyimpan features sebagai **Dynamic Table** yang bisa di-versioning, di-discover, dan di-reuse oleh multiple models.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session,
    database="BANK_DB",
    name="ML_FEATURES",
    default_warehouse="BANK_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

nasabah_entity = Entity(
    name="NASABAH",
    join_keys=["NASABAH_ID"],
    desc="Entity nasabah bank untuk credit scoring"
)

fs.register_entity(nasabah_entity)
print("Entity NASABAH registered!")
print(f"\nEntities: {fs.list_entities().to_pandas()}")

In [ ]:
trx_fv_df = trx_features.select(
    "NASABAH_ID", "AVG_TRX_AMOUNT", "TOTAL_TRX_COUNT",
    "DIGITAL_CHANNEL_RATIO", "FAILED_TRX_RATIO", "TOTAL_TRX_VOLUME"
)

trx_feature_view = FeatureView(
    name="NASABAH_TRANSACTION_FEATURES",
    entities=[nasabah_entity],
    feature_df=trx_fv_df,
    desc="Features transaksi per nasabah: avg amount, count, digital ratio, failed ratio"
)

trx_feature_view = fs.register_feature_view(
    feature_view=trx_feature_view,
    version="V1",
    overwrite=True
)
print("Transaction FeatureView V1 registered!")

simpanan_fv_df = simpanan_features.select(
    "NASABAH_ID", "TOTAL_SIMPANAN", "JUMLAH_PRODUK_SIMPANAN",
    "MAX_SALDO_SIMPANAN", "AVG_BUNGA_SIMPANAN"
)

simpanan_feature_view = FeatureView(
    name="NASABAH_SIMPANAN_FEATURES",
    entities=[nasabah_entity],
    feature_df=simpanan_fv_df,
    desc="Features simpanan per nasabah: total saldo, jumlah produk, max saldo"
)

simpanan_feature_view = fs.register_feature_view(
    feature_view=simpanan_feature_view,
    version="V1",
    overwrite=True
)
print("Simpanan FeatureView V1 registered!")

print(f"\nFeature Views:\n{fs.list_feature_views().to_pandas()[['NAME', 'VERSION', 'DESC']]}")

---
## Part 3: Model Training - XGBoost Credit Scoring

Kita akan train **XGBoost Classifier** untuk credit scoring karena:
- XGBoost sangat baik untuk tabular data & imbalanced dataset
- Memiliki built-in handling untuk missing values
- Menghasilkan feature importance yang penting untuk explainability (regulasi OJK)
- Performa terbukti di industri perbankan

### 3.1 Prepare Training Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

pdf = master_features.to_pandas()

categorical_cols = ['JENIS_KREDIT', 'SEKTOR_EKONOMI', 'AGUNAN', 'SEGMEN_NASABAH', 'PEKERJAAN', 'JENIS_KELAMIN']
label_encoders = {}
for c in categorical_cols:
    le = LabelEncoder()
    pdf[c] = le.fit_transform(pdf[c].astype(str))
    label_encoders[c] = le

feature_cols = [
    'JUMLAH_PINJAMAN', 'OUTSTANDING', 'TENOR_BULAN', 'SUKU_BUNGA_PERSEN',
    'CREDIT_UTILIZATION_RATIO',
    'JENIS_KREDIT', 'SEKTOR_EKONOMI', 'AGUNAN',
    'PENGHASILAN_BULANAN', 'SEGMEN_NASABAH', 'PEKERJAAN', 'JENIS_KELAMIN',
    'LAMA_JADI_NASABAH_TAHUN', 'USIA',
    'AVG_TRX_AMOUNT', 'TOTAL_TRX_COUNT', 'DIGITAL_CHANNEL_RATIO',
    'FAILED_TRX_RATIO', 'TOTAL_TRX_VOLUME',
    'TOTAL_SIMPANAN', 'JUMLAH_PRODUK_SIMPANAN', 'MAX_SALDO_SIMPANAN',
    'SAVING_RATIO'
]

X = pdf[feature_cols].fillna(0)
y = pdf['IS_DEFAULT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set    : {X_test.shape[0]:,} samples")
print(f"Default rate (train): {y_train.mean()*100:.1f}%")
print(f"Default rate (test) : {y_test.mean()*100:.1f}%")

### 3.2 Train XGBoost - Baseline Model (V1)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_v1 = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False
)

model_v1.fit(X_train, y_train)

y_pred_v1 = model_v1.predict(X_test)
y_prob_v1 = model_v1.predict_proba(X_test)[:, 1]

auc_v1 = roc_auc_score(y_test, y_prob_v1)
ap_v1 = average_precision_score(y_test, y_prob_v1)

print("=" * 50)
print("BASELINE MODEL (V1) - Results")
print("=" * 50)
print(f"\nAUC-ROC Score : {auc_v1:.4f}")
print(f"Average Precision: {ap_v1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_v1, target_names=['Non-Default', 'Default']))

### 3.3 Hyperparameter Tuning - Model V2

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False
)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV AUC-ROC: {grid_search.best_score_:.4f}")

In [ ]:
model_v2 = grid_search.best_estimator_

y_pred_v2 = model_v2.predict(X_test)
y_prob_v2 = model_v2.predict_proba(X_test)[:, 1]

auc_v2 = roc_auc_score(y_test, y_prob_v2)
ap_v2 = average_precision_score(y_test, y_prob_v2)

print("=" * 50)
print("TUNED MODEL (V2) - Results")
print("=" * 50)
print(f"\nAUC-ROC Score : {auc_v2:.4f}")
print(f"Average Precision: {ap_v2:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_v2, target_names=['Non-Default', 'Default']))

print(f"\n{'='*50}")
print(f"COMPARISON: V1 vs V2")
print(f"{'='*50}")
print(f"AUC-ROC  -> V1: {auc_v1:.4f} | V2: {auc_v2:.4f} | Improvement: {(auc_v2-auc_v1)*100:.2f}%")
print(f"Avg Prec -> V1: {ap_v1:.4f}  | V2: {ap_v2:.4f}  | Improvement: {(ap_v2-ap_v1)*100:.2f}%")

### 3.4 Visualisasi Model Performance

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fpr_v1, tpr_v1, _ = roc_curve(y_test, y_prob_v1)
fpr_v2, tpr_v2, _ = roc_curve(y_test, y_prob_v2)
axes[0].plot(fpr_v1, tpr_v1, label=f'V1 Baseline (AUC={auc_v1:.3f})', color='#3498db', linewidth=2)
axes[0].plot(fpr_v2, tpr_v2, label=f'V2 Tuned (AUC={auc_v2:.3f})', color='#e74c3c', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

precision_v2, recall_v2, _ = precision_recall_curve(y_test, y_prob_v2)
axes[1].plot(recall_v2, precision_v2, color='#e74c3c', linewidth=2)
axes[1].set_title(f'Precision-Recall Curve (AP={ap_v2:.3f})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(True, alpha=0.3)

importance = model_v2.feature_importances_
sorted_idx = np.argsort(importance)[-15:]
axes[2].barh(range(len(sorted_idx)), importance[sorted_idx], color='#2ecc71')
axes[2].set_yticks(range(len(sorted_idx)))
axes[2].set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=9)
axes[2].set_title('Top 15 Feature Importance', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Importance')

plt.suptitle('Credit Scoring Model - Performance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
# plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
# print("Plot saved: model_performance.png")

---
## Part 4: Model Registry & Deployment

Snowflake Model Registry menyediakan:
- **Versioning:** Track multiple versi model (V1 baseline, V2 tuned)
- **Governance:** Siapa yang membuat model, kapan, dari data apa
- **Deployment:** Langsung inference via SQL (warehouse) atau REST API (SPCS)
- **Lineage:** Trace dari source data → features → model

### 4.1 Register Model V1 (Baseline) ke Registry

In [ ]:
from snowflake.ml.registry import Registry

session.sql("CREATE SCHEMA IF NOT EXISTS BANK_DB.ML_MODELS COMMENT = 'Schema untuk ML Model Registry'").collect()

reg = Registry(session=session, database_name="BANK_DB", schema_name="ML_MODELS")

sample_input = X_train.head(10)

mv_v1 = reg.log_model(
    model_v1,
    model_name="CREDIT_SCORING_UMKM",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost", "scikit-learn"],
    target_platforms=["WAREHOUSE"],
    metrics={
        "auc_roc": float(auc_v1),
        "average_precision": float(ap_v1),
        "model_type": "XGBoost Baseline",
    },
    comment="Baseline XGBoost Credit Scoring Model untuk Kredit UMKM - no tuning"
)

print(f"Model V1 registered: {mv_v1.model_name} version {mv_v1.version_name}")
print(f"AUC-ROC: {auc_v1:.4f}")

### 4.2 Register Model V2 (Tuned) ke Registry

In [ ]:
def _encode_snowpark(col_name, le):
    classes = list(le.classes_)
    expr = F.when(F.col(col_name) == F.lit(classes[0]), F.lit(0))
    for i, cls in enumerate(classes[1:], start=1):
        expr = expr.when(F.col(col_name) == F.lit(cls), F.lit(i))
    return expr.otherwise(F.lit(-1)).cast(T.IntegerType()).alias(col_name)

select_exprs = []
for c in feature_cols:
    if c in label_encoders:
        select_exprs.append(_encode_snowpark(c, label_encoders[c]))
    else:
        select_exprs.append(F.col(c).alias(c))

sample_sp_df = master_features.select(*select_exprs).na.fill(0).limit(10)

print("Snowpark sample columns:", sample_sp_df.columns)
sample_sp_df.show(3)

In [ ]:
try:
    reg.get_model("CREDIT_SCORING_UMKM").delete_version("V2")
    print("Existing V2 dropped.")
except Exception as e:
    print(f"No existing V2 to drop (ok): {e}")

mv_v2 = reg.log_model(
    model_v2,
    model_name="CREDIT_SCORING_UMKM",
    version_name="V2",
    sample_input_data=sample_sp_df,
    conda_dependencies=["xgboost", "scikit-learn"],
    target_platforms=["WAREHOUSE"],
    metrics={
        "auc_roc": float(auc_v2),
        "average_precision": float(ap_v2),
        "model_type": "XGBoost Tuned (GridSearchCV)",
        "best_params": str(grid_search.best_params_),
    },
    comment="Tuned XGBoost Credit Scoring Model - re-logged with Snowpark sample_input for lineage"
)

print(f"Model V2 re-registered: {mv_v2.model_name} version {mv_v2.version_name}")
print(f"AUC-ROC: {auc_v2:.4f}")

### 4.3 Verifikasi Model di Registry

In [ ]:
models_df = session.sql("SHOW MODELS IN SCHEMA BANK_DB.ML_MODELS").collect()
print("=== Registered Models ===")
for row in models_df:
    print(f"  - {row['name']}")

print("\n=== Model Functions (V2) ===")
functions_df = session.sql(
    "SHOW FUNCTIONS IN MODEL BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM VERSION V2"
).collect()
for row in functions_df:
    print(f"  - {row['name']}")

print("\n=== Model Metrics ===")
v1_metrics = mv_v1.show_metrics()
v2_metrics = mv_v2.show_metrics()

print("V1 Metrics:")
for k, v in v1_metrics.items():
    print(f"  - {k}: {v}")

print("\nV2 Metrics:")
for k, v in v2_metrics.items():
    print(f"  - {k}: {v}")

---
## Part 5: Batch Inference & Explainability (SHAP)

### 5.1 Batch Inference via SQL

Setelah model ter-register, kita bisa langsung scoring via SQL query - tanpa perlu Python!

In [ ]:
encoded_features = pdf[["KREDIT_ID", "NASABAH_ID", "IS_DEFAULT"] + feature_cols].copy()

session.write_pandas(
    encoded_features,
    table_name="CREDIT_SCORING_FEATURES",
    database="BANK_DB",
    schema="RAW_DATA",
    auto_create_table=True,
    overwrite=True,
)
print("Feature table saved (encoded): CREDIT_SCORING_FEATURES")

feature_col_list = ", ".join(feature_cols)

batch_sql = f"""
SELECT
    KREDIT_ID,
    NASABAH_ID,
    IS_DEFAULT AS ACTUAL_DEFAULT,
    MODEL(BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM, V2)!PREDICT(
        {feature_col_list}
    ):output_feature_0::INT AS PREDICTED_DEFAULT,
    MODEL(BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM, V2)!PREDICT_PROBA(
        {feature_col_list}
    ):output_feature_1::FLOAT AS DEFAULT_PROBABILITY
FROM BANK_DB.RAW_DATA.CREDIT_SCORING_FEATURES
LIMIT 20
"""

print("=== Batch Inference Results (Top 20) ===")
result = session.sql(batch_sql)
result.show()

### 5.2 Model Explainability - SHAP Values

> **WAJIB untuk regulasi OJK:** Model harus bisa menjelaskan kenapa approve/reject kredit nasabah.

Snowflake Model Registry otomatis menghasilkan `EXPLAIN` method yang menggunakan SHAP (SHapley Additive exPlanations).

In [ ]:
explain_sql = f"""
WITH MV_ALIAS AS MODEL BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM VERSION V2
SELECT
    t.KREDIT_ID,
    t.NASABAH_ID,
    t.IS_DEFAULT,
    e.*
FROM BANK_DB.RAW_DATA.CREDIT_SCORING_FEATURES t,
    TABLE(MV_ALIAS!EXPLAIN({feature_col_list})) e
LIMIT 10
"""

print("=== SHAP Explanations (Top 10) ===")
print("Setiap kolom *_explanation menunjukkan kontribusi feature terhadap prediksi")
explain_result = session.sql(explain_sql)
explain_result.show()

In [ ]:
shap_pd = explain_result.to_pandas()
explanation_cols = [c for c in shap_pd.columns if c.endswith('_EXPLANATION')]

mean_abs_shap = shap_pd[explanation_cols].abs().mean().sort_values(ascending=True)
mean_abs_shap.index = [c.replace('_EXPLANATION', '') for c in mean_abs_shap.index]

fig, ax = plt.subplots(figsize=(10, 8))
top_n = mean_abs_shap.tail(15)
ax.barh(range(len(top_n)), top_n.values, color='#e74c3c', edgecolor='white')
ax.set_yticks(range(len(top_n)))
ax.set_yticklabels(top_n.index, fontsize=10)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('Feature Importance (SHAP) - Credit Scoring Model', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
# plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
# print("Plot saved: shap_importance.png")
print("\nInterpretasi: Features dengan SHAP value tinggi memiliki pengaruh besar terhadap keputusan kredit")

---
## Part 6: ML Observability & Monitoring

### 6.1 Setup Model Monitor

ML Observability memungkinkan kita memonitor:
- **Prediction Drift:** Apakah distribusi prediksi berubah seiring waktu?
- **Feature Drift:** Apakah distribusi input features berubah?
- **Performance Metrics:** Track accuracy, precision, recall secara berkala

In [ ]:
feature_col_list = ", ".join(feature_cols)

session.sql(f"""
CREATE OR REPLACE TABLE BANK_DB.RAW_DATA.CREDIT_SCORING_PREDICTIONS AS
SELECT
    KREDIT_ID,
    NASABAH_ID,
    IS_DEFAULT AS ACTUAL_DEFAULT,
    {feature_col_list},
    MODEL(BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM, V2)!PREDICT_PROBA(
        {feature_col_list}
    ):output_feature_1::FLOAT AS DEFAULT_PROBABILITY,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS PREDICTION_TS
FROM BANK_DB.RAW_DATA.CREDIT_SCORING_FEATURES
""").collect()
print("Inference log table created: CREDIT_SCORING_PREDICTIONS")

session.sql("""
CREATE OR REPLACE MODEL MONITOR BANK_DB.ML_MODELS.CREDIT_SCORING_MONITOR
WITH
    MODEL = BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM
    VERSION = V2
    FUNCTION = 'predict_proba'
    WAREHOUSE = BANK_WH
    SOURCE = BANK_DB.RAW_DATA.CREDIT_SCORING_PREDICTIONS
    TIMESTAMP_COLUMN = PREDICTION_TS
    PREDICTION_SCORE_COLUMNS = (DEFAULT_PROBABILITY)
    ACTUAL_CLASS_COLUMNS = (ACTUAL_DEFAULT)
    ID_COLUMNS = (KREDIT_ID)
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
""").collect()

print("Model Monitor created: BANK_DB.ML_MODELS.CREDIT_SCORING_MONITOR")
print("Monitor tracks: prediction drift, feature drift, and performance (vs ACTUAL_DEFAULT)")

print(session.sql("SHOW MODEL MONITORS IN SCHEMA BANK_DB.ML_MODELS").to_pandas())

### 6.2 ML Lineage - Audit Trail

ML Lineage sangat penting untuk audit OJK: *"Dari mana data model ini berasal?"*

In [ ]:
print("=== ML Lineage: Credit Scoring Model V2 ===")
print()
print("Data Sources:")
print("  ├── BANK_DB.RAW_DATA.DIM_NASABAH (10,000 nasabah)")
print("  ├── BANK_DB.RAW_DATA.DIM_CABANG (67 cabang)")
print("  ├── BANK_DB.RAW_DATA.FACT_KREDIT (5,000 kredit)")
print("  ├── BANK_DB.RAW_DATA.FACT_TRANSAKSI (100,000 transaksi)")
print("  └── BANK_DB.RAW_DATA.FACT_SIMPANAN (15,000 simpanan)")
print()
print("Feature Store:")
print("  ├── BANK_DB.ML_FEATURES.NASABAH_TRANSACTION_FEATURES (V1)")
print("  └── BANK_DB.ML_FEATURES.NASABAH_SIMPANAN_FEATURES (V1)")
print()
print("Model Registry:")
print("  ├── BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM (V1 - Baseline)")
print("  └── BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM (V2 - Tuned) ← Active")
print()
print("Inference:")
print("  └── SQL: MODEL(CREDIT_SCORING_UMKM, V2)!PREDICT(...)")

try:
    upstream = mv_v2.lineage(direction="upstream")
    downstream = mv_v2.lineage(direction="downstream")

    print("\n=== Upstream Lineage (data sources feeding the model) ===")
    if upstream:
        for node in upstream:
            print(f"  - {node}")
    else:
        print("  (no upstream nodes found)")

    print("\n=== Downstream Lineage (objects derived from the model) ===")
    if downstream:
        for node in downstream:
            print(f"  - {node}")
    else:
        print("  (no downstream nodes found)")
except Exception as e:
    print(f"\nNote: Lineage query returned: {e}")
    print("(Lineage tracking requires the model to be logged with Snowpark DataFrame as sample_input_data)")

### 6.3 Simulasi Model Drift Detection

In [ ]:
from scipy import stats

np.random.seed(42)
original_probs = y_prob_v2
shifted_probs = np.clip(original_probs + np.random.normal(0.05, 0.02, len(original_probs)), 0, 1)

ks_stat, ks_pvalue = stats.ks_2samp(original_probs, shifted_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(original_probs, bins=50, alpha=0.7, color='#3498db', label='Baseline (Week 1)', density=True)
axes[0].hist(shifted_probs, bins=50, alpha=0.7, color='#e74c3c', label='Current (Week 4)', density=True)
axes[0].set_title('Prediction Distribution Drift', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Default Probability')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

weeks = ['Week 1', 'Week 2', 'Week 3', 'Week 4']
auc_over_time = [auc_v2, auc_v2 - 0.005, auc_v2 - 0.015, auc_v2 - 0.03]
threshold = 0.75

axes[1].plot(weeks, auc_over_time, 'o-', color='#2ecc71', linewidth=2, markersize=8, label='AUC-ROC')
axes[1].axhline(y=threshold, color='#e74c3c', linestyle='--', linewidth=2, label=f'Threshold ({threshold})')
axes[1].set_title('Model Performance Over Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(max(0.0, min(min(auc_over_time), threshold) - 0.05), min(1.0, max(max(auc_over_time), threshold) + 0.05))

plt.suptitle('ML Observability - Drift Detection', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
# plt.savefig('drift_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nKS Test Statistic: {ks_stat:.4f}")
print(f"KS Test P-Value  : {ks_pvalue:.6f}")
if ks_pvalue < 0.05:
    print("⚠️  ALERT: Significant drift detected! Consider retraining the model.")
else:
    print("✅ No significant drift detected.")

# print(f"\nPlot saved: drift_detection.png")

---
## Cleanup (Opsional - Setelah Workshop)

```sql
-- Hapus model
DROP MODEL IF EXISTS BANK_DB.ML_MODELS.CREDIT_SCORING_UMKM;

-- Hapus feature store objects
DROP SCHEMA IF EXISTS BANK_DB.ML_FEATURES CASCADE;
DROP SCHEMA IF EXISTS BANK_DB.ML_MODELS CASCADE;
DROP TABLE IF EXISTS BANK_DB.RAW_DATA.CREDIT_SCORING_FEATURES;
```

---
## Summary

| Komponen | Detail |
|----------|--------|
| **Use Case** | Credit Scoring untuk Kredit UMKM |
| **Target Variable** | IS_DEFAULT (binary: kolektibilitas 3-5 = Default) |
| **Algorithm** | XGBoost Classifier |
| **Feature Store** | 2 Feature Views (Transaction + Simpanan) |
| **Model Registry** | V1 (Baseline) + V2 (Tuned) |
| **Inference** | Batch via SQL: `MODEL()!PREDICT()` |
| **Explainability** | SHAP via `MODEL()!EXPLAIN()` |
| **Monitoring** | Prediction drift, feature drift, performance tracking |
| **Lineage** | Source tables → Features → Model → Predictions |

**Koneksi ke Regulasi OJK:**
- Model harus **explainable** (SHAP) → sesuai prinsip transparansi
- Data harus **traceable** (ML Lineage) → sesuai prinsip akuntabilitas
- Model harus **monitored** (Drift Detection) → sesuai prinsip keamanan

**Selamat! Anda telah berhasil membangun Credit Scoring Model end-to-end di Snowflake!**